# Orden Topológico

**Sesión de programación competitiva — grafos**

## ¿Qué es un orden topológico?

Dado un grafo dirigido $G = (V, E)$, un **orden topológico** es una secuencia de todos sus vértices $v_1, v_2, \dots, v_n$ tal que, para toda arista dirigida $(u, v) \in E$, el vértice $u$ aparece antes que $v$ en la secuencia. Dicho de otra forma: nada puede aparecer antes que sus propios requisitos.

Un orden así **solo existe si $G$ es un DAG** (*Directed Acyclic Graph*, grafo dirigido y acíclico). Si el grafo tiene un ciclo, ningún orden puede respetar todas las aristas a la vez — y detectar esa imposibilidad es, en la práctica, la mitad de los problemas de esta familia.

**¿Dónde aparece en contest?** Prerrequisitos de cursos, orden de compilación de módulos, resolución de dependencias de paquetes, planificación de tareas con restricciones, y — como en los tres problemas de hoy — decidir si un proceso por etapas (cocinar, imprimir, construir) es siquiera posible.

Existen dos algoritmos clásicos para calcular un orden topológico, ambos en $O(V + E)$:

1. **Algoritmo de Kahn** (BFS por grado de entrada).
2. **DFS con postorden invertido**.

Vamos a recorrer los dos usando el mismo grafo de ejemplo.


* Imágenes: https://leetcode.com/discuss/post/1078072/introduction-to-topological-sort-by-sinc-i0ii/

### Grafo de ejemplo

<img src="Figure 1.jpeg" width="700" alt="Grafo dirigido de ejemplo y un orden topologico valido">

*A la izquierda, el DAG de ejemplo. A la derecha, uno de sus posibles órdenes topológicos, linealizado (con las aristas que "saltan" nodos dibujadas como arcos).*


In [1]:
# Grafo de la Figura 1
GRAPH = {1: [4, 2], 2: [3], 3: [], 4: [2, 5, 6], 5: [6], 6: []}
GRAPH


{1: [4, 2], 2: [3], 3: [], 4: [2, 5, 6], 5: [6], 6: []}

## Algoritmo de Kahn (BFS)

**Idea:** un nodo puede procesarse en cuanto ya no le falta ningún prerrequisito, es decir, en cuanto su grado de entrada (`indeg`) llega a 0.

```
Kahn(grafo):
    para cada nodo v en grafo:
        indeg[v] = número de aristas que entran a v

    cola = todos los nodos v con indeg[v] == 0
    orden = []

    mientras cola no esté vacía:
        u = extraer un nodo de la cola
        orden.append(u)
        para cada arista (u, w):
            indeg[w] -= 1
            si indeg[w] == 0:
                cola.push(w)

    si len(orden) < n:
        el grafo tiene un ciclo -> no existe orden topológico
    si no:
        orden es un orden topológico válido
```

> **Truco de detección de ciclos:** si al terminar `len(orden) < n`, sobraron nodos cuyo grado de entrada nunca bajó a 0 — forman un ciclo. Este truco es exactamente la idea detrás del **Problema A**.

Para desempatar cuándo hay varios nodos disponibles al mismo tiempo, aquí usamos un **min-heap**: siempre extraemos el nodo disponible con el id más chico, lo que produce el orden topológico lexicográficamente más pequeño.


In [4]:
import heapq

def kahn_topological_sort(graph):
    indegree = {u: 0 for u in graph}
    for u in graph:
        for v in graph[u]:
            indegree[v] += 1

    heap = [u for u in graph if indegree[u] == 0]
    heapq.heapify(heap)

    order = []
    steps = []  # (nodo_procesado, orden_acumulado_hasta_ahora)
    while heap:
        u = heapq.heappop(heap)
        order.append(u)
        for v in graph[u]:
            indegree[v] -= 1
            if indegree[v] == 0:
                heapq.heappush(heap, v)
        steps.append((u, list(order)))

    if len(order) != len(graph):
        raise ValueError("El grafo tiene un ciclo: no existe orden topológico")
    return order, steps

kahn_order, kahn_steps = kahn_topological_sort(GRAPH)
print("Orden topológico (Kahn):", kahn_order)


Orden topológico (Kahn): [1, 4, 2, 3, 5, 6]


### Traza de Kahn paso a paso

<img src="Figure 2.jpeg" width="800" alt="Seis pasos del algoritmo de Kahn sobre el grafo de ejemplo">

*Los nodos en rosa ya salieron de la cola y fueron agregados al arreglo de resultado. El arreglo final, `[1, 4, 2, 3, 5, 6]`, coincide exactamente con `kahn_order` de la celda anterior.*


## Alternativa: DFS con postorden invertido

**Idea:** si recorremos el grafo en profundidad y anotamos cada nodo en el momento en que *terminamos* de explorarlo (postorden), todo nodo termina **después** de todos sus descendientes. Invirtiendo esa lista de "terminados" obtenemos un orden topológico válido.

```
DFS_topologico(grafo):
    visitado = {}
    pila_finalizacion = []

    funcion dfs(u):
        visitado[u] = true
        para cada arista (u, w):
            si w no está visitado:
                dfs(w)
        pila_finalizacion.push(u)      // u termina aquí

    para cada nodo u en grafo:
        si u no está visitado:
            dfs(u)

    orden = invertir(pila_finalizacion)
```

> **Detección de ciclos con DFS:** un `visitado` simple no alcanza — hace falta marcar también qué nodos están *en la pila de recursión actual* (color gris). Si al explorar los vecinos de `u` llegamos a un nodo gris, encontramos una arista de retroceso ⇒ hay un ciclo.

Con el mismo grafo, el orden que produce DFS **no tiene por qué coincidir** con el de Kahn — ambos son válidos, porque puede existir más de un orden topológico correcto para un mismo DAG.


In [5]:
def dfs_topological_sort(graph):
    visited = set()
    finish_stack = []
    steps = []  # (nodo_terminado, pila_de_finalizacion_hasta_ahora)

    def dfs(u):
        visited.add(u)
        for v in sorted(graph[u]):
            if v not in visited:
                dfs(v)
        finish_stack.append(u)
        steps.append((u, list(finish_stack)))

    for u in sorted(graph):
        if u not in visited:
            dfs(u)

    return list(reversed(finish_stack)), steps

dfs_order, dfs_steps = dfs_topological_sort(GRAPH)
print("Pila de finalización (orden real de terminado):", [s[0] for s in dfs_steps])
print("Orden topológico (DFS, pila invertida):", dfs_order)


Pila de finalización (orden real de terminado): [3, 2, 6, 5, 4, 1]
Orden topológico (DFS, pila invertida): [1, 4, 5, 6, 2, 3]


### Traza de DFS paso a paso

<img src="Figure 3.jpeg" width="800" alt="Recorrido DFS mostrando el orden de finalizacion de cada nodo">

*Cada panel resalta el nodo "Current" y va llenando la pila de finalización desde la derecha. Esta traza visita a los vecinos en un orden distinto al de `dfs_topological_sort` (y usa un grafo sin la arista 4→2), así que el resultado final que ves aquí (`1, 2, 3, 4, 5, 6`) puede no coincidir con el que imprime la celda de código — ambos son órdenes topológicos válidos, solo cambia qué vecino se visita primero.*


## Complejidad y resumen

| | Kahn (BFS) | DFS + postorden |
|---|---|---|
| Complejidad | $O(V + E)$ | $O(V + E)$ |
| Detecta ciclos | `len(orden) < n` | nodo gris revisitado |
| Orden que produce | el lexicográficamente mínimo (con heap) | depende del orden de visita |

---

## Problemas de hoy

Tres problemas adaptados a formato ICPC para practicar orden topológico. Cada uno tiene su propia forma de modelar el grafo — identifícala antes de escribir código.


### Problem A — Study Plan

**Time limit:** 2 seconds &nbsp;&nbsp; **Memory limit:** 256 MB

**Statement**

The Faculty of Sciences offers `n` courses, numbered from `0` to `n-1`. There are `m` prerequisite relations: the pair `(a, b)` means that, in order to enroll in course `a`, you must have already passed course `b`.

A student wants to know whether it is possible to design a study plan that lets them take **all** `n` courses while respecting every prerequisite. Help them find out.

**Input**

The first line contains two integers `n` and `m`. Each of the next `m` lines contains two integers `a b`, meaning course `a` requires course `b`.

**Output**

Print `YES` if a valid study plan covering all `n` courses exists, or `NO` otherwise.

**Constraints**

- `1 <= n <= 2 * 10^5`
- `0 <= m <= 5 * 10^5`
- `0 <= a, b < n`, `a != b`
- no pair `(a, b)` is repeated

**Example 1**

Input:
```
4 4
1 0
2 0
3 1
3 2
```
Output:
```
YES
```

**Example 2**

Input:
```
2 2
1 0
0 1
```
Output:
```
NO
```
Course 1 requires course 0, and course 0 requires course 1 — a cycle of size 2, impossible to complete.


### Problem B — Shelter Recipe Book

**Time limit:** 2 seconds &nbsp;&nbsp; **Memory limit:** 256 MB

**Statement**

A mountain shelter has `k` raw supplies available and a recipe book with `n` recipes. Each recipe has a name and a list of ingredients; an ingredient can be either a raw supply **or the result of preparing another recipe** from the same book.

A recipe can be prepared only when **all** of its ingredients are already available (as a raw supply, or because the recipe that produces them was already prepared). Determine which recipes can eventually be prepared, and output them in a valid order in which they could actually be cooked one after another.

**Input**

The first line contains an integer `n`. Then follow `n` blocks, one per recipe: a line with the recipe's name and an integer `t` (number of ingredients), followed by a line with the `t` ingredient names. After that comes an integer `k` and a line with the `k` names of the available raw supplies.

**Output**

Print `c`, the number of recipes that can be prepared, followed on the next line by their names separated by spaces, in a valid preparation order. If `c = 0`, print only the line with `0`.

**Constraints**

- `1 <= n <= 100`
- each recipe has between 1 and 10 ingredients
- `1 <= k <= 100`
- names are unique across recipes and supplies, and contain only lowercase letters
- no recipe (directly or indirectly) requires itself

**Example**

Input:
```
3
bread 2
yeast flour
sandwich 2
bread meat
burger 3
sandwich meat bread
3
yeast flour meat
```
Output:
```
3
bread sandwich burger
```
With yeast, flour and meat available, `bread` can be made; with bread and meat, `sandwich`; and with sandwich, meat and bread, `burger`. This is exactly a topological order over the recipe -> ingredient-recipe graph.

> **Hint:** model each recipe as a node with `indeg` = number of ingredients that are *still* unprepared recipes. Seed the queue with recipes whose `indeg` reaches 0 once raw supplies are accounted for, and when a recipe finishes cooking, treat it as a newly available supply that may unlock others.


### Problem C — The Layered Mural

**Time limit:** 2 seconds &nbsp;&nbsp; **Memory limit:** 256 MB

**Statement**

An artist paints a mural on an `n`-row by `m`-column grid by placing, one after another, rectangular tarps of a single solid color: each tarp fully covers an axis-aligned rectangle with one color, and a later tarp may fully or partially cover earlier ones.

At the end of the process only the topmost color at each cell is visible, giving the final grid `grid[i][j]`. Given that grid, determine whether **some** order of tarps could have produced it.

**Input**

The first line contains two integers `n` and `m`. Each of the next `n` lines contains `m` integers: the grid `grid[i][j]`, where each value is a color id.

**Output**

Print `YES` if the grid is reachable by painting rectangular tarps in some order, or `NO` otherwise.

**Constraints**

- `1 <= n, m <= 20`
- `1 <= grid[i][j] <= 60`

**Example 1**

Input:
```
5 4
1 1 1 1
1 2 2 1
1 2 1 1
1 2 2 2
1 1 1 1
```
Output:
```
YES
```
Paint the full 5x4 rectangle with color 1 first, then the rectangle of color 2 on top. No color needs to be painted before itself.

**Example 2**

Input:
```
3 3
1 2 1
2 1 2
1 2 1
```
Output:
```
NO
```
This is a checkerboard pattern: the minimal bounding rectangle of color 1 also contains cells of color 2, and vice versa. Each color would need to be painted after the other — a dependency cycle of size 2.

> **Hint:** for each color, compute the minimal bounding rectangle containing all its cells. If a cell of a different color appears inside that rectangle, that tarp must have been painted *before* the other color: add an edge `color -> other color`. The grid is reachable if and only if this dependency graph between colors has **no cycle** — exactly what a topological sort lets you check.
